# Variance Analysis on Multi-task MLP Probing

Analyze the multi-task MLP probing results (`mlp_probing_crc_multi`) by gene variance.

**Setup:**
- Frozen MorphPT features (gate 2.5×+10×)
- Shared MLP(768→512→400), LayerNorm
- Same 400 genes (top coverage)
- Spatial split
- Result: mean_r = 0.262

**Question:** Is variance still the strongest predictor in the multi-task setup?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.ndimage import uniform_filter1d

PROJECT = Path('/hpc/group/jilab/tc459/MorphPT')
CACHE   = PROJECT / 'cache_crc'
MULTI_DIR = PROJECT / 'experiments/mlp_probing_crc_multi'
PERGENE_DIR = PROJECT / 'experiments/mlp_probing_crc'

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False,
})

## 1. Load Multi-task Results

In [ ]:
multi_df = pd.read_csv(MULTI_DIR / 'multi_mlp_results.csv')
print(f'Multi-task results: {len(multi_df)} genes')
print(f'Mean test Pearson: {multi_df["test_pearson"].mean():.4f}')
print(f'Median test Pearson: {multi_df["test_pearson"].median():.4f}')
multi_df.head()

## 2. Load Per-gene Results for Comparison

In [ ]:
# Collect per-gene results
import json
rows = []
for d in sorted(PERGENE_DIR.iterdir()):
    if not d.is_dir():
        continue
    f = d / 'results.json'
    if not f.exists():
        continue
    r = json.loads(f.read_text())
    rows.append({
        'gene_name':        r.get('gene_name', d.name),
        'gene_idx':         r.get('gene_idx', -1),
        'pergene_test_r':   r.get('test_pearson', 0),
    })
per_df = pd.DataFrame(rows)
print(f'Per-gene results: {len(per_df)} genes')
print(f'Mean per-gene test Pearson: {per_df["pergene_test_r"].mean():.4f}')

## 3. Compute Gene Statistics

In [ ]:
expr = np.load(str(CACHE / 'expr.npy'), mmap_mode='r')
meta = pd.read_csv(CACHE / 'meta.csv')
train_idx = np.sort(meta[meta['split']=='train']['mmap_idx'].values)

print(f'Computing stats on {len(train_idx):,} train cells...')

n_genes = expr.shape[1]
sum_x   = np.zeros(n_genes, dtype=np.float64)
sum_x2  = np.zeros(n_genes, dtype=np.float64)
sum_nz  = np.zeros(n_genes, dtype=np.float64)

batch = 5000
for i in range(0, len(train_idx), batch):
    X = expr[train_idx[i:i+batch]].astype(np.float64)
    sum_x  += X.sum(axis=0)
    sum_x2 += (X**2).sum(axis=0)
    sum_nz += (X > 0).sum(axis=0)

n_train       = len(train_idx)
gene_mean     = sum_x / n_train
gene_std      = np.sqrt(np.clip(sum_x2/n_train - gene_mean**2, 0, None))
gene_cv       = gene_std / np.clip(gene_mean, 1e-5, None)
gene_coverage = sum_nz / n_train

print(f'Done.')

In [ ]:
# Merge stats into multi_df
multi_df['std_expr']  = multi_df['gene_idx'].map(dict(zip(range(n_genes), gene_std)))
multi_df['mean_expr'] = multi_df['gene_idx'].map(dict(zip(range(n_genes), gene_mean)))
multi_df['cv']        = multi_df['gene_idx'].map(dict(zip(range(n_genes), gene_cv)))
multi_df['coverage']  = multi_df['gene_idx'].map(dict(zip(range(n_genes), gene_coverage)))

# Merge per-gene results
multi_df = multi_df.merge(per_df[['gene_name', 'pergene_test_r']],
                          on='gene_name', how='left')

# Rename for clarity
multi_df = multi_df.rename(columns={'test_pearson': 'multi_test_r',
                                     'val_pearson':  'multi_val_r'})

df = multi_df.dropna(subset=['std_expr']).copy()
print(f'After merge: {len(df)} genes with stats')
df[['gene_name', 'coverage', 'std_expr',
    'pergene_test_r', 'multi_test_r']].head(10)

## 4. Correlation Analysis

In [ ]:
print(f'Correlation between gene stats and MULTI-TASK test Pearson r:')
print(f'{"="*60}')
print(f'{"Metric":<20} {"Pearson":>10} {"Spearman":>10}')
print('-' * 60)
for col, label in [('coverage', 'Coverage'),
                   ('mean_expr', 'Mean expression'),
                   ('std_expr', 'Std expression'),
                   ('cv', 'CV')]:
    r, _    = stats.pearsonr(df[col], df['multi_test_r'])
    r_sp, _ = stats.spearmanr(df[col], df['multi_test_r'])
    print(f'{label:<20} {r:>+10.4f} {r_sp:>+10.4f}')

print(f'\nFor reference — PER-GENE test Pearson r:')
for col, label in [('coverage', 'Coverage'),
                   ('std_expr', 'Std expression')]:
    r, _ = stats.pearsonr(df[col], df['pergene_test_r'])
    print(f'  {label:<18} {r:>+10.4f}')

## 5. Variance Quintile Analysis

In [ ]:
df['var_quintile'] = pd.qcut(df['std_expr'], q=5,
                              labels=['Q1 (lowest)', 'Q2',
                                      'Q3 (median)', 'Q4',
                                      'Q5 (highest)'])

summary = df.groupby('var_quintile', observed=True).agg(
    n_genes          = ('multi_test_r',  'count'),
    std_min          = ('std_expr',      'min'),
    std_max          = ('std_expr',      'max'),
    coverage_mean    = ('coverage',      'mean'),
    pergene_mean_r   = ('pergene_test_r','mean'),
    multi_mean_r     = ('multi_test_r',  'mean'),
    multi_median_r   = ('multi_test_r',  'median'),
    multi_frac_gt_02 = ('multi_test_r',  lambda s: (s > 0.2).mean()),
    multi_frac_gt_03 = ('multi_test_r',  lambda s: (s > 0.3).mean()),
).round(3)

# Delta (multi - per-gene)
summary['delta'] = (summary['multi_mean_r'] -
                    summary['pergene_mean_r']).round(4)

print('Multi-task MLP performance by variance quintile:')
print(summary.to_string())

print(f'\nQ1 multi r : {summary.iloc[0]["multi_mean_r"]:.3f}')
print(f'Q5 multi r : {summary.iloc[-1]["multi_mean_r"]:.3f}')
delta_q5_q1 = summary.iloc[-1]['multi_mean_r'] - summary.iloc[0]['multi_mean_r']
print(f'Q5-Q1 delta: {delta_q5_q1:+.3f}')

## 6. Key Question: top N by variance?

In [ ]:
df_sorted = df.sort_values('std_expr', ascending=False).reset_index(drop=True)

print(f'If we had selected top N genes by VARIANCE:')
print(f'{"="*70}')
print(f'{"N":>5} {"multi_r":>10} {"pergene_r":>12} {"r>0.2":>8} {"r>0.3":>8}')
print('-' * 60)
for n in [100, 150, 200, 250, 300, 400]:
    top_n = df_sorted.head(n)
    print(f'{n:>5} '
          f'{top_n["multi_test_r"].mean():>10.4f} '
          f'{top_n["pergene_test_r"].mean():>12.4f} '
          f'{(top_n["multi_test_r"]>0.2).mean():>8.1%} '
          f'{(top_n["multi_test_r"]>0.3).mean():>8.1%}')

print(f'\nCurrent (all 400): multi={df["multi_test_r"].mean():.4f}  '
      f'pergene={df["pergene_test_r"].mean():.4f}')

## 7. Multi-task vs Per-gene: Where does multi-task help?

In [ ]:
df['delta'] = df['multi_test_r'] - df['pergene_test_r']

print(f'Overall delta (multi - per-gene):')
print(f'  mean  : {df["delta"].mean():+.4f}')
print(f'  median: {df["delta"].median():+.4f}')
print(f'  multi > per-gene: {(df["delta"] > 0).sum()} / {len(df)} genes '
      f'({(df["delta"] > 0).mean():.1%})')

print(f'\nBy variance quintile:')
print(f'{"Quintile":<15} {"pergene_r":>10} {"multi_r":>10} {"delta":>8}')
print('-' * 48)
for q in summary.index:
    sub = df[df['var_quintile'] == q]
    print(f'{str(q):<15} '
          f'{sub["pergene_test_r"].mean():>10.4f} '
          f'{sub["multi_test_r"].mean():>10.4f} '
          f'{sub["delta"].mean():>+8.4f}')

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Scatter: variance vs multi_test_r
ax = axes[0]
sc = ax.scatter(df['std_expr'], df['multi_test_r'],
                c=df['coverage'], cmap='viridis',
                s=12, alpha=0.6, rasterized=True)
plt.colorbar(sc, ax=ax, label='Coverage')

sort_idx = np.argsort(df['std_expr'].values)
window   = max(10, len(df)//20)
smooth   = uniform_filter1d(df['multi_test_r'].values[sort_idx], size=window)
ax.plot(df['std_expr'].values[sort_idx], smooth, color='black', lw=2)

r, _ = stats.pearsonr(df['std_expr'], df['multi_test_r'])
ax.text(0.05, 0.95, f'Pearson r = {r:+.3f}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

ax.axhline(0,   color='gray', lw=0.8, ls='--')
ax.axhline(0.3, color='green', lw=0.6, ls=':', alpha=0.6)
ax.set_xlabel('Std of expression')
ax.set_ylabel('Multi-task test Pearson r')
ax.set_title('(a) Multi-task r vs variance')

# (b) Quintile comparison: per-gene vs multi-task
ax = axes[1]
x = np.arange(5)
w = 0.35
pg_means    = [df[df['var_quintile']==q]['pergene_test_r'].mean()
               for q in summary.index]
multi_means = [df[df['var_quintile']==q]['multi_test_r'].mean()
               for q in summary.index]

b1 = ax.bar(x - w/2, pg_means, w, label='Per-gene MLP',
            color='#4C72B0', alpha=0.8)
b2 = ax.bar(x + w/2, multi_means, w, label='Multi-task MLP',
            color='#C44E52', alpha=0.8)
for bar in b1:
    ax.annotate(f'{bar.get_height():.3f}',
                (bar.get_x()+bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', fontsize=7)
for bar in b2:
    ax.annotate(f'{bar.get_height():.3f}',
                (bar.get_x()+bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([str(q).replace(' ', '\n') for q in summary.index],
                   fontsize=7)
ax.set_ylabel('Mean Pearson r')
ax.set_title('(b) Per-gene vs Multi-task by variance quintile')
ax.legend()
ax.axhline(0.3, color='green', lw=0.6, ls=':', alpha=0.5)

# (c) Scatter: per-gene vs multi-task
ax = axes[2]
sc = ax.scatter(df['pergene_test_r'], df['multi_test_r'],
                c=df['std_expr'], cmap='viridis',
                s=10, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='Std of expression')

lims = [-0.05, max(df['pergene_test_r'].max(),
                   df['multi_test_r'].max()) + 0.05]
ax.plot(lims, lims, 'k--', lw=0.8, label='y = x')

ax.set_xlabel('Per-gene MLP test r')
ax.set_ylabel('Multi-task MLP test r')
ax.set_title(f'(c) Multi-task vs Per-gene\n'
             f'mean delta = {df["delta"].mean():+.4f}')
ax.legend(fontsize=8)

fig.suptitle('Variance analysis: Multi-task MLP Probing (CRC, 400 genes)',
             fontweight='bold', fontsize=12)
fig.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print(f'{"="*60}')
print('MULTI-TASK MLP PROBING — VARIANCE ANALYSIS SUMMARY')
print(f'{"="*60}')
print()
print(f'All 400 genes:')
print(f'  Per-gene MLP   : {df["pergene_test_r"].mean():.4f}')
print(f'  Multi-task MLP : {df["multi_test_r"].mean():.4f}')
print(f'  Delta          : {df["delta"].mean():+.4f}')
print()
print(f'Variance effect (Q5 vs Q1):')
q1 = summary.iloc[0]
q5 = summary.iloc[-1]
print(f'  Per-gene Q1→Q5 : {q1["pergene_mean_r"]:.3f} → {q5["pergene_mean_r"]:.3f}')
print(f'  Multi-task Q1→Q5: {q1["multi_mean_r"]:.3f} → {q5["multi_mean_r"]:.3f}')
print()

top200 = df_sorted.head(200)
print(f'If we had used top 200 genes by variance:')
print(f'  Multi-task mean r: {top200["multi_test_r"].mean():.4f}')
print(f'  r > 0.3: {(top200["multi_test_r"] > 0.3).mean():.1%}')
print()

# Save merged analysis
out_csv = PROJECT / 'analysis/multi_vs_pergene_variance.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print(f'Saved → {out_csv}')